# 🚀 SUSA-1B CodeMind — Master Google Colab GPU Pipeline

> **An original coding-focused Transformer language model built and trained from scratch under the SUSA technology ecosystem.**

---

### 📌 Zero-Cost Engineering Principles:
- **Hardware Cost:** ₹0 (leveraging Google Colab Free Tier GPUs — NVIDIA T4 / V100).
- **Core Model:** Real decoder-only Transformer with RoPE, SwiGLU, RMSNorm, Grouped-Query Attention (GQA), and tied embeddings.
- **Persistent Checkpoints:** High-speed NVMe writes locally synced continuously to Google Drive for disconnection recovery.
- **Full Stack:** Ingestion, Pretraining, Instruction Tuning (SFT with Prompt Loss Masking), Sandboxed Evaluation, and Inference.

## Step 1: Connect to GitHub & Clone / Update Repository

In [ ]:
#@title Step 1: Clone / Update Repository from GitHub { display-mode: "form" }
import os

WORKSPACE_DIR = "/content/susa-1b-codemind"
REPO_URL = "https://github.com/saketh-nandu/susa-1b-codemind.git" #@param {type:"string"}
BRANCH = "main" #@param {type:"string"}

if os.path.exists(WORKSPACE_DIR):
    print(f"Directory {WORKSPACE_DIR} already exists. Pulling latest code...")
    %cd {WORKSPACE_DIR}
    !git checkout {BRANCH}
    !git pull origin {BRANCH}
else:
    print(f"Cloning {REPO_URL} into {WORKSPACE_DIR}...")
    !git clone --branch {BRANCH} {REPO_URL} {WORKSPACE_DIR}
    %cd {WORKSPACE_DIR}

!pwd
!git log -n 1 --oneline

## Step 2: Install Dependencies & Setup Environment

In [ ]:
#@title Step 2: Install Dependencies { display-mode: "form" }
!pip install --upgrade pip
!pip install -e ".[colab,dev]"
!pip install psutil pyyaml tokenizers tqdm bitsandbytes datasets


## Step 3: Inspect GPU Hardware & Environment Diagnostics

In [ ]:
#@title Step 3: Check GPU Hardware { display-mode: "form" }
import torch
from susacodemind.utils.system_checker import SystemInfo

sys_info = SystemInfo.inspect()
print(sys_info.summary())

if not sys_info.cuda_available:
    print("\n" + "!" * 70)
    print("  [CRITICAL WARNING] No CUDA GPU detected in this Google Colab session!")
    print("  To train with high throughput, enable the free GPU accelerator:")
    print("  1. Go to top menu: Runtime -> Change runtime type")
    print("  2. Hardware accelerator: Select 'T4 GPU'")
    print("  3. Click 'Save' and re-run this notebook.")
    print("!" * 70)
else:
    device_name = torch.cuda.get_device_name(0)
    vram_gb = sys_info.gpu_devices[0]['total_vram_gb'] if sys_info.gpu_devices else 0.0
    print(f"\n[OK] CUDA GPU acceleration ACTIVE: {device_name} ({vram_gb:.2f} GiB VRAM)")

## Step 4: Mount Google Drive (Persistent Storage & Crash Recovery)

In [ ]:
#@title Step 4: Mount Google Drive { display-mode: "form" }
ENABLE_DRIVE_SYNC = True #@param {type:"boolean"}
DRIVE_DIR = "/content/drive/MyDrive/SUSA_CodeMind_Checkpoints" #@param {type:"string"}

if ENABLE_DRIVE_SYNC:
    from google.colab import drive
    print("Authorizing Google Drive mount...")
    drive.mount("/content/drive")
    os.makedirs(DRIVE_DIR, exist_ok=True)
    print(f"Persistent checkpoint storage ready at: {DRIVE_DIR}")
else:
    print("Drive synchronization disabled. Checkpoints will be kept in local runtime NVMe.")

## Step 5: Select Architecture Profile & Inspect Parameter Count

In [ ]:
#@title Step 5: Select Architecture Profile { display-mode: "form" }
PROFILE = "target_1b" #@param ["target_1b", "medium", "small", "smoke"]

from susacodemind.model.config import ModelConfig
from susacodemind.utils.resource_estimator import estimate_resources

cfg_path = f"configs/{PROFILE}.yaml"
model_cfg = ModelConfig.from_yaml(cfg_path)
print(model_cfg.print_parameter_summary())

# Feasibility calculation with 8-bit Adam for 1B model
res = estimate_resources(model_cfg, batch_size=model_cfg.training.batch_size, seq_len=model_cfg.max_seq_len, use_8bit_adam=True)
print(res.summary())


## Step 6: Run Comprehensive Automated Test Suite (All 49 Tests)

In [ ]:
#@title Step 6: Run Pytest Test Suite { display-mode: "form" }
!pytest tests/ -v

## Step 7: Acquire Clean Code Dataset & SFT Instructions

In [ ]:
#@title Step 7: Download Online Free Coding Datasets & SFT Instructions { display-mode: "form" }
# Ingests EVERYTHING: Hugging Face (the-stack-smol), GitHub top repos (TheAlgorithms, requests, flask, fastapi, jinja, rich), and Python stdlib
DATA_SOURCE = "all" #@param ["all", "huggingface", "github", "stdlib"]
HF_SAMPLES = 5000 #@param {type:"integer"}

!python scripts/download_dataset.py --source {DATA_SOURCE} --hf-samples {HF_SAMPLES} --output-dir data/raw_code --generate-instructions --instructions-file data/coding_instructions.jsonl


## Step 8: Train Byte-Level BPE Tokenizer from Scratch

In [ ]:
#@title Step 8: Train BPE Tokenizer { display-mode: "form" }
TOKENIZER_DIR = f"tokenizer_cache/{PROFILE}"
VOCAB_SIZE = model_cfg.vocab_size

if not os.path.exists(os.path.join(TOKENIZER_DIR, "tokenizer.json")):
    print(f"Training tokenizer for profile '{PROFILE}' with target vocab {VOCAB_SIZE:,}...")
    if os.path.exists("data/raw_code"):
        !python scripts/train_tokenizer.py --input-dir data/raw_code --vocab-size {VOCAB_SIZE} --output-dir {TOKENIZER_DIR}
    else:
        !python scripts/train_tokenizer.py --use-synthetic --vocab-size {VOCAB_SIZE} --output-dir {TOKENIZER_DIR}
else:
    print(f"Tokenizer already exists at {TOKENIZER_DIR}.")

## Step 9: Sanitize, Scrub Secrets, Deduplicate, & Shard Binary Dataset

In [ ]:
#@title Step 9: Ingest & Shard Dataset { display-mode: "form" }
if os.path.exists("data/raw_code"):
    !python scripts/prepare_dataset.py --tokenizer-dir {TOKENIZER_DIR} --input-dir data/raw_code --output-dir data
else:
    !python scripts/prepare_dataset.py --tokenizer-dir {TOKENIZER_DIR} --use-synthetic --output-dir data

## Step 10: Launch Pretraining with Periodic Checkpointing & AMP

In [ ]:
#@title Step 10: Launch Pretraining (Target 1B Model) { display-mode: "form" }
PRETRAIN_STEPS = 1000 #@param {type:"integer"}
BATCH_SIZE = 1 #@param {type:"integer"}
RESUME_CHECKPOINT = "" #@param {type:"string"}

train_cmd = f"python scripts/train.py --config configs/{PROFILE}.yaml --train-bin data/train.bin --val-bin data/val.bin --max-steps {PRETRAIN_STEPS} --batch-size {BATCH_SIZE}"
if RESUME_CHECKPOINT.strip():
    train_cmd += f" --resume-from {RESUME_CHECKPOINT.strip()}"

print(f"Executing: {train_cmd}")
!{train_cmd}


## Step 11: Supervised Instruction Fine-Tuning (SFT with Prompt Loss Masking)

In [ ]:
#@title Step 11: Instruction Fine-Tuning (SFT on Real Coding Tasks) { display-mode: "form" }
import glob

available_ckpts = sorted(glob.glob("checkpoints/*.pt"))
base_ckpt = available_ckpts[-1] if available_ckpts else ""
INSTRUCT_STEPS = 100 #@param {type:"integer"}

if base_ckpt:
    print(f"Fine-tuning 1B base checkpoint: {base_ckpt}...")
    !python scripts/train_instruct.py --base-checkpoint {base_ckpt} --tokenizer-dir {TOKENIZER_DIR} --data-file data/coding_instructions.jsonl --max-steps {INSTRUCT_STEPS} --output-dir checkpoints/instruct
else:
    print("No base checkpoint available to instruction-tune. Please run Step 10 first.")


## Step 12: Benchmark Evaluation (Pass@1 Sandbox & Perplexity)

In [ ]:
#@title Step 12: Run Benchmark Evaluation { display-mode: "form" }
eval_ckpt = "checkpoints/instruct/checkpoint_instruct_final.pt"
if not os.path.exists(eval_ckpt) and available_ckpts:
    eval_ckpt = available_ckpts[-1]

if os.path.exists(eval_ckpt):
    print(f"Evaluating checkpoint: {eval_ckpt}...")
    !python scripts/evaluate.py --checkpoint {eval_ckpt} --tokenizer-dir {TOKENIZER_DIR} --val-bin data/val.bin
else:
    print("No checkpoint available for evaluation.")

## Step 13: Interactive Code Generation

In [ ]:
#@title Step 13: Generate Code { display-mode: "form" }
PROMPT_TEXT = "def quicksort(arr: list[int]) -> list[int]:\n" #@param {type:"string"}
TEMPERATURE = 0.3 #@param {type:"slider", min:0.0, max:1.0, step:0.05}
MAX_TOKENS = 64 #@param {type:"integer"}

if os.path.exists(eval_ckpt):
    !python scripts/generate.py --checkpoint {eval_ckpt} --tokenizer-dir {TOKENIZER_DIR} --prompt "{PROMPT_TEXT}" --max-new-tokens {MAX_TOKENS} --temperature {TEMPERATURE}

## Step 14: Sync Checkpoints to Google Drive & Export Run Manifest

In [ ]:
#@title Step 14: Sync Checkpoints to Google Drive { display-mode: "form" }
if ENABLE_DRIVE_SYNC and os.path.exists("/content/drive/MyDrive"):
    print(f"Syncing checkpoints to Google Drive: {DRIVE_DIR}...")
    !python scripts/sync_checkpoints.py --src-dir checkpoints --dest-dir "{DRIVE_DIR}"
    if os.path.exists("checkpoints/instruct"):
        !python scripts/sync_checkpoints.py --src-dir checkpoints/instruct --dest-dir "{DRIVE_DIR}/instruct"
else:
    print("Drive synchronization disabled or Drive not mounted.")

## Step 15: Execution Summary & Zero-Cost Audit

In [ ]:
#@title Step 15: Execution Summary { display-mode: "form" }
saved_ckpts = sorted(glob.glob("checkpoints/**/*.pt", recursive=True))
print("=" * 70)
print("  SUSA-1B CODEMIND — COMPLETE RUN SUMMARY")
print("=" * 70)
print(f"  Profile:              {PROFILE}")
print(f"  Parameters:           {model_cfg.calculate_exact_parameters()['formatted_total']}")
print(f"  Saved Checkpoints:    {len(saved_ckpts)}")
for c in saved_ckpts:
    print(f"    - {c}")
if ENABLE_DRIVE_SYNC:
    print(f"  Google Drive Backup:  {DRIVE_DIR}")
print("  Total Compute Cost:   ₹0 (Free Google Colab Session)")
print("=" * 70)
print("All SUSA-1B CodeMind milestones executed successfully!")